# Hackathon 1 — Asistente Inteligente con Llama
## Asistente del laboratorio de microalgas

> **Propósito:** construir un asistente funcional que responda preguntas frecuentes sobre el protocolo de cultivo de microalgas del laboratorio, usando RAG sobre el documento del protocolo y Llama para generar la respuesta.

### Esta entrega demuestra

- Problema y usuario claramente definidos.
- Solución funcional basada en Llama.
- Estrategia de prompting documentada.
- Base de conocimiento y flujo RAG.
- Embeddings y recuperación semántica.
- Generación con Llama usando contexto recuperado.
- Pruebas normales y fuera de alcance.
- Evaluación de resultados.
- Decisiones técnicas, alcance y siguientes pasos.
- Pipeline y documentación suficiente para reproducir el proyecto.

> **Nota:** El Hackathon permite distintos enfoques: RAG, fine-tuning con LoRA o una combinación. Este proyecto elige **RAG** porque el problema requiere responder con información propia del protocolo de cultivo. La decisión de no aplicar fine-tuning queda documentada.


# 1. Definición del proyecto

## Problema
Los estudiantes y miembros del laboratorio preguntan una y otra vez lo mismo sobre el cultivo de microalgas: qué medio usar, cómo medir el crecimiento, a qué temperatura cultivar y cómo cosechar la biomasa. Las respuestas están en el protocolo de cultivo, pero nadie lo consulta.

## Usuario objetivo
Estudiantes y miembros del Laboratorio de Biotecnología Integral y Bioinformática (UASLP, FCQ) que trabajan con cultivos de *Chlorella vulgaris* y *Scenedesmus acutus*.

## Solución
Construir un asistente basado en Llama que recupere información del protocolo de cultivo y genere respuestas breves basadas en ese contexto.

## Criterio de éxito
El asistente debe:
1. recuperar el fragmento pertinente del protocolo;
2. responder usando ese contexto;
3. evitar inventar un dato del protocolo cuando la información no está en la base;
4. mostrar evidencia de pruebas realizadas.


## 1.1 Decisión de arquitectura

### Estrategia elegida: RAG

El conocimiento que necesitamos responder pertenece a un documento específico del laboratorio: el protocolo de cultivo.

```text
Pregunta del usuario
        ↓
Embedding de la pregunta
        ↓
Búsqueda semántica
        ↓
Fragmento relevante
        ↓
Prompt con contexto
        ↓
Llama
        ↓
Respuesta
```

### ¿Por qué no hacemos fine-tuning en este ejemplo?

No necesitamos modificar el comportamiento interno del modelo para incorporar el protocolo de cultivo. El objetivo principal es que Llama responda utilizando información externa específica.

Por eso documentamos el fine-tuning como una **decisión no utilizada**, en lugar de presentarlo como una etapa ficticia.


# 2. Preparación del entorno

Instalamos las librerías necesarias y comprobamos los recursos disponibles.

- `transformers`: carga y ejecuta el modelo.
- `accelerate`: facilita la ejecución del modelo.
- `sentence-transformers`: genera embeddings.
- `numpy`: operaciones con vectores.
- `torch`: entorno de ejecución.


In [ ]:
!pip install transformers accelerate sentence-transformers --quiet

import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 3. Configuración de modelos

Usaremos:

- **TinyLlama/TinyLlama-1.1B-Chat-v1.0** para generación.
- **paraphrase-multilingual-MiniLM-L12-v2** para embeddings.

El modelo de embeddings no genera respuestas: representa los textos como vectores para comparar su similitud semántica.


In [ ]:
MODELO_LLAMA = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODELO_EMBEDDINGS = "paraphrase-multilingual-MiniLM-L12-v2"

print("Modelo generativo:", MODELO_LLAMA)
print("Modelo de embeddings:", MODELO_EMBEDDINGS)


# 4. Cargar TinyLlama

Cargamos el tokenizer y el modelo generativo.

**Resultado esperado:** TinyLlama queda disponible para la etapa de generación.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_LLAMA)

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_LLAMA,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("TinyLlama cargado correctamente")
print("Dispositivo:", modelo.device)


# 5. Preparación de la base de conocimiento

Esta base representa el contenido que, en un proyecto real, provendría del protocolo de cultivo del laboratorio.

Los fragmentos resumen el protocolo que se usa en el laboratorio para *Chlorella vulgaris* y *Scenedesmus acutus*: medio BG-11, medición de crecimiento por OD680, condiciones de luz y temperatura, cosecha e inóculo.


In [ ]:
documentos = [
    "MEDIO DE CULTIVO: Chlorella vulgaris se cultiva en medio BG-11, a 25 °C, con fotoperiodo "
    "de 12 horas de luz y 12 horas de oscuridad, y aireación constante.",
    "MEDICIÓN DE CRECIMIENTO: el crecimiento del cultivo se sigue por densidad óptica a 680 nm "
    "(OD680); se mide cada 24 horas en espectrofotómetro, usando medio BG-11 estéril como blanco.",
    "COSECHA DE BIOMASA: la biomasa se cosecha por centrifugación a 4,000 rpm durante 15 minutos; "
    "el pellet se lava con agua destilada y se seca a 60 °C hasta peso constante.",
    "LUZ: la luz azul favorece la acumulación de lípidos y la luz roja el crecimiento; "
    "el fotoperiodo de trabajo es 12 horas de luz y 12 horas de oscuridad.",
    "CONTAMINACIÓN: si un cultivo se contamina, se descarta, se esteriliza el material "
    "y se vuelve a inocular desde un cultivo axénico.",
    "INÓCULO: el inóculo se prepara a partir de un cultivo en fase exponencial, "
    "con una densidad óptica inicial (OD680) de 0.1.",
]

print("Número de fragmentos:", len(documentos))
for i, doc in enumerate(documentos, 1):
    print(f"{i}. {doc}")


# 6. Indexación: generación de embeddings

Convertimos cada fragmento de la base de conocimiento en un vector.

Esta es la etapa de **indexación** del RAG.


In [ ]:
modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

embeddings_documentos = modelo_embeddings.encode(
    documentos,
    normalize_embeddings=True
)

print("Forma de los embeddings:", embeddings_documentos.shape)


# 7. Retrieval: búsqueda semántica

La función recibe una pregunta, genera su embedding y calcula su similitud con cada fragmento.

Devolvemos el fragmento recuperado, su posición y el score de similitud.


In [ ]:
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        normalize_embeddings=True
    )[0]

    similitudes = np.dot(
        embeddings_documentos,
        embedding_pregunta
    )

    indice = int(np.argmax(similitudes))

    return {
        "indice": indice,
        "fragmento": documentos[indice],
        "score": float(similitudes[indice])
    }


## 7.1 Prueba aislada del retrieval

Antes de incorporar Llama, comprobamos que la búsqueda semántica esté recuperando información razonable.

Primero validamos componentes individuales y después el flujo completo.


In [ ]:
preguntas_retrieval = [
    "¿Cómo mido el crecimiento de mi cultivo?",
    "¿A qué temperatura cultivo Chlorella?",
    "¿Cómo cosecho la biomasa?",
    "¿Qué hago si mi cultivo se contamina?",
]

for pregunta in preguntas_retrieval:
    resultado = buscar_fragmento(pregunta)
    print("\nPregunta:", pregunta)
    print("Fragmento:", resultado["fragmento"])
    print("Score:", round(resultado["score"], 4))


# 8. Prompt Engineering

El prompt establece el comportamiento del asistente.

La regla central es:

> **Responder únicamente con la información proporcionada en el contexto.**

Si el contexto no contiene la respuesta, el asistente debe reconocer que no cuenta con información suficiente en lugar de inventarla.


In [ ]:
SYSTEM_PROMPT = """
Eres un asistente del laboratorio de microalgas.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes valores, condiciones ni procedimientos.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.
"""


# 9. Generación con Llama

La función recibe un prompt completo y genera la respuesta.

El modelo recibe instrucciones, contexto recuperado y pregunta del usuario.


In [ ]:
def generar_respuesta(prompt):
    mensajes = [
        {"role": "user", "content": prompt}
    ]

    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto,
        return_tensors="pt"
    ).to(modelo.device)

    salida = modelo.generate(
        **entradas,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(
        salida[0],
        skip_special_tokens=True
    )


# 10. Pipeline completo del asistente

Integramos:

**pregunta → retrieval → prompt → Llama → respuesta**

La función también devuelve el contexto y el score para que la ejecución pueda auditarse durante las pruebas.


In [ ]:
def asistente(pregunta):
    recuperacion = buscar_fragmento(pregunta)

    prompt = f"""
{SYSTEM_PROMPT}

Contexto recuperado:
{recuperacion["fragmento"]}

Pregunta del usuario:
{pregunta}

Respuesta:
"""

    respuesta = generar_respuesta(prompt)

    return {
        "pregunta": pregunta,
        "contexto": recuperacion["fragmento"],
        "score": recuperacion["score"],
        "respuesta": respuesta
    }


# 11. Demo funcional

Esta es una demostración de extremo a extremo: una pregunta real atraviesa todo el pipeline y produce una respuesta.


In [ ]:
pregunta_demo = "¿Cómo mido el crecimiento de mi cultivo?"

resultado = asistente(pregunta_demo)

print("PREGUNTA")
print(resultado["pregunta"])

print("\nCONTEXTO RECUPERADO")
print(resultado["contexto"])

print("\nSCORE")
print(round(resultado["score"], 4))

print("\nRESPUESTA")
print(resultado["respuesta"])


# 12. Evaluación del asistente

Una entrega de Hackathon debe mostrar pruebas, no únicamente una demostración favorable.

Usaremos:

1. **Caso conocido:** la respuesta está en la base.
2. **Caso conocido con otra formulación:** comprobamos la búsqueda semántica.
3. **Caso fuera de alcance:** la información no está en la base y esperamos una respuesta controlada.


In [ ]:
casos_prueba = [
    {
        "id": "TC01",
        "pregunta": "¿Cómo sigo el crecimiento de mi cultivo?",
        "esperado": "680 nm"
    },
    {
        "id": "TC02",
        "pregunta": "¿A qué temperatura debo cultivar Chlorella vulgaris?",
        "esperado": "25 °C"
    },
    {
        "id": "TC03",
        "pregunta": "¿Cómo cosecho la biomasa del cultivo?",
        "esperado": "4,000 rpm"
    },
    {
        "id": "TC04",
        "pregunta": "¿Cuál es el horario de la biblioteca?",
        "esperado": "no hay información"
    }
]


## 12.1 Ejecutar las pruebas

Para cada caso mostramos la pregunta, el contexto recuperado y la respuesta generada.


In [ ]:
resultados_pruebas = []

for caso in casos_prueba:
    resultado = asistente(caso["pregunta"])

    registro = {
        "id": caso["id"],
        "pregunta": caso["pregunta"],
        "esperado": caso["esperado"],
        "contexto": resultado["contexto"],
        "score": resultado["score"],
        "respuesta": resultado["respuesta"]
    }

    resultados_pruebas.append(registro)

    print("=" * 80)
    print(caso["id"])
    print("Pregunta:", caso["pregunta"])
    print("Esperado:", caso["esperado"])
    print("Contexto:", resultado["contexto"])
    print("Respuesta:", resultado["respuesta"])


# 13. Evaluación resumida

La evaluación distingue entre:

- **recuperación:** ¿se recuperó el fragmento correcto?
- **generación:** ¿la respuesta respeta el contexto?
- **fuera de alcance:** ¿evita inventar información?

Los criterios son sencillos para que el equipo pueda inspeccionarlos manualmente.


In [ ]:
def evaluar_resultado(resultado, criterio):
    texto = resultado["respuesta"].lower()

    if criterio == "680 nm":
        return "680" in texto

    if criterio == "25 °C":
        return "25" in texto

    if criterio == "4,000 rpm":
        return "4,000" in texto or "4000" in texto

    if criterio == "no hay información":
        return any(
            frase in texto
            for frase in [
                "no cuento con",
                "no tengo información",
                "no está en el contexto",
                "no se especifica"
            ]
        )

    return False


evaluacion = []

for caso, resultado in zip(casos_prueba, resultados_pruebas):
    ok = evaluar_resultado(resultado, caso["esperado"])
    evaluacion.append({
        "id": caso["id"],
        "resultado": "OK" if ok else "REVISAR"
    })

print("RESULTADOS DE EVALUACIÓN")
for item in evaluacion:
    print(item["id"], "->", item["resultado"])


# 14. Alcance y siguientes pasos

### Alcance de este prototipo

- La base de conocimiento es pequeña y está escrita manualmente a partir del protocolo.
- El retrieval recupera un único fragmento.
- No existe una base vectorial persistente.
- La evaluación es pequeña.
- TinyLlama es un modelo pequeño y puede producir respuestas imperfectas.
- El prototipo corre en Colab y no representa por sí mismo un despliegue productivo.

### Riesgos

- Recuperación incorrecta.
- Respuestas que no respeten completamente el contexto.
- Información insuficiente para responder.
- Dependencia del entorno de ejecución.

### Mejora siguiente

Ampliar la base documental con el protocolo completo, mejorar el chunking, recuperar varios fragmentos y ampliar el conjunto de pruebas.


# 15. Pipeline del proyecto

| Etapa | Implementación |
|---|---|
| Datos | Fragmentos del protocolo de cultivo de microalgas |
| Preparación | Organización de los fragmentos |
| Recuperación | Embeddings + similitud semántica |
| Generación | TinyLlama |
| Prompting | Instrucciones + contexto recuperado |
| Evaluación | Casos de prueba y criterios explícitos |
| Despliegue | **Prototipo en Colab** |
| Alcance | No se presenta Colab como producción |

### Decisión sobre fine-tuning

No se aplicó LoRA porque el objetivo principal es incorporar conocimiento documental específico, problema que RAG aborda directamente.

La decisión queda documentada para distinguir entre una técnica no necesaria y una técnica omitida accidentalmente.


# 16. Reproducibilidad

Esta entrega deja documentados:

- modelo generativo;
- modelo de embeddings;
- dependencias;
- fuente y contenido de la base de conocimiento;
- estrategia de prompting;
- algoritmo de retrieval;
- pipeline de generación;
- casos de prueba;
- criterios de evaluación;
- alcance.

> **Para una entrega real:** el equipo debe sustituir este ejemplo por su propio problema, datos, decisiones técnicas y resultados.


# 17. Checklist de entrega del Hackathon 1

- [ ] Problema y usuario objetivo definidos.
- [ ] Asistente funcional basado en Llama.
- [ ] Estrategia de prompting documentada.
- [ ] Fuente de conocimiento documentada si se usa RAG.
- [ ] Embeddings y retrieval ejecutables.
- [ ] Respuesta basada en contexto recuperado.
- [ ] Pruebas con preguntas reales.
- [ ] Caso fuera de alcance o fallo controlado.
- [ ] Resultados de evaluación.
- [ ] Decisiones técnicas explicadas.
- [ ] Alcance documentado.
- [ ] Pipeline completo documentado.
- [ ] Decisión sobre fine-tuning documentada.
- [ ] Notebook reproducible.

## Evidencia para la presentación

El equipo debería poder mostrar:

1. el problema;
2. el asistente funcionando;
3. la estrategia de prompting;
4. RAG o la estrategia técnica elegida;
5. las pruebas;
6. los resultados;
7. decisiones y alcance.


# 18. Conclusión

Este notebook representa una **entrega completa**, no solamente una demo del modelo.

```text
Problema
   ↓
Datos
   ↓
Estrategia técnica
   ↓
RAG / Prompt
   ↓
Llama
   ↓
Pruebas
   ↓
Evaluación
   ↓
Alcance
   ↓
Entrega y presentación
```

El objetivo del Hackathon es demostrar que el equipo puede convertir lo aprendido durante el Módulo 1 en un **asistente funcional, probado y presentable**.
